# Data Retrieval and Pre-Processing -- Overnight Stays

In [1]:
import pandas as pd
import geopandas as gpd 
import matplotlib.pyplot as plt
import numpy as np
from shapely.validation import make_valid
import warnings

In [2]:
# suppress warning not needed that crowd the output
warnings.filterwarnings("ignore", module="shapely.constructive", category=RuntimeWarning)

## Reading in relevant data

### first reading in the data with number of overnight stays from Statistics Iceland

In [3]:
# reading in the excel from Iceland Statistics

overnight_df_2024 = pd.read_excel('Overnight_stays_per_muncip_Iceland_2024.xlsx')

In [4]:
overnight_df_2024.head()

,Allt landið,Reykjavík,Kópavogur,Hafnarfjörður,Mosfellsbær,Kjósarhreppur,Reykjanesbær,Grindavík,Sveitarfélagið Vogar,Akraneskaupstaður,...,Hveragerði,Sveitarfélagið Ölfus,Grímsnes- og Grafningshreppur,Skeiða- og Gnúpverjahreppur,Bláskógabyggð,Garðabær,Suðurnesjabær,Ásahreppur,Flóahreppur,Seltjarnarnes
0,9312648,3218190,85385,94045,34910,..,413463,41282,5237,34131,...,147147,53704,42686,23138,207200,..,104339,3657,29399,388


In [5]:
# transpose for easier processing

overnight_df_2024_long = overnight_df_2024.melt(var_name="region", value_name="nights")

In [6]:
overnight_df_2024_long.head(n=10)

,region,nights
0,Allt landið,9312648
1,Reykjavík,3218190
2,Kópavogur,85385
3,Hafnarfjörður,94045
4,Mosfellsbær,34910
5,Kjósarhreppur,..
6,Reykjanesbær,413463
7,Grindavík,41282
8,Sveitarfélagið Vogar,5237
9,Akraneskaupstaður,34131


In [7]:
len(overnight_df_2024_long)

63

In [8]:
# check data type of number of nights for the suspicious looking one at index 5
type(overnight_df_2024_long.loc[5]['nights'])

str

In [9]:
# clean up those that do not contain a number by changing those with strings to NaN

overnight_df_2024_long['nights'] = pd.to_numeric(overnight_df_2024_long['nights'],errors='coerce')

In [10]:
# there are two muncipalities with missing values
sum(overnight_df_2024_long['nights'].isna())

2

After some consideration, I have to decided to simply keep these municipalities in there and make clear in the visualization that they have missing data, as I find that most informative and easiest to work with.

### retrieve relevant geoinformation using LMI WFS

In [11]:
# LMÍ WFS endpoint for municipal boundaries
url = "https://gis.lmi.is/geoserver/wfs?service=WFS&version=2.0.0&request=GetFeature&typeName=IS_50V:mork_sveitarf_flakar&outputFormat=application/json"
gdf = gpd.read_file(url)

In [12]:
print(gdf.crs)
print(gdf.shape)

EPSG:4326
(69, 19)


In [13]:
gdf.head(n=2)

,id,objectid,uuid,fitjuflokkur,stjornsyslusvaedi,nrsveitarfelags,sveitarfelag,upphafnotkunar,ath,vinnsluferlifitju,dagsheimildar,dagsinnsetningar,dagsleidrettingar,dagsuppfaerslu,nakvaemnixy,heimild,gagnaeigandi,gagnasafn,geometry
0,mork_sveitarf_flakar.87,87,733aab03-8fe4-d216-8337-3af9272a0928,101,4,4604,Vesturbyggð,2024-05-19,,4,2024-05-19,2003-01-01,2024-06-20,2024-06-26,50,LMÍ,LMÍ,"IS 50V, útgáfa 26.06.2024","MULTIPOLYGON (((-23.53210 65.72040, -23.53150 ..."
1,mork_sveitarf_flakar.54,54,94b0003e-2bc6-e4ad-3a10-cc25e8cb5139,101,4,6513,Eyjafjarðarsveit,1991-01-01,,4,NaN,2003-01-01,2018-08-23,2024-06-26,50,LMÍ,LMÍ,"IS 50V, útgáfa 26.06.2024","MULTIPOLYGON (((-18.02490 65.67850, -18.02490 ..."


### read in population data from Statistics Iceland, to be used later on

In [14]:
# read in population data 
population_df_2024 = pd.read_excel('population_per_mucipality_iceland_2024.xlsx', header=None)

In [15]:
population_df_2024.head()

,0,1
0,Reykjavikurborg,136894
1,Kopavogsbaer,39335
2,Seltjarnarnesbaer,4572
3,Gardabaer,19088
4,Hafnarfjardarkaupstadur,30616


In [16]:
# rename columns
population_df_2024 = population_df_2024.rename(columns = {0: "municipality", 1: "population"})

In [17]:
population_df_2024.head(2)

,municipality,population
0,Reykjavikurborg,136894
1,Kopavogsbaer,39335


In [18]:
len(population_df_2024)

62

## Pre-Processing

### ensure the dataset contain the same (amount of) municipalities

In [19]:
# since there are 69 rows, but Iceland only has 62 or 63 muncipalities (depends on what year you look at and how the count is done), 
# we need to check out what is going on

In [20]:
print(gdf['sveitarfelag'].value_counts())  

sveitarfelag
Grímsnes- og Grafningshreppur    3
Akureyrarbær                     2
Hafnarfjarðarkaupstaður          2
Ísafjarðarbær                    2
Kópavogsbær                      2
                                ..
Tjörneshreppur                   1
Húnaþing vestra                  1
Skagafjörður                     1
Eyjafjarðarsveit                 1
Fljótsdalshreppur                1
Name: count, Length: 62, dtype: int64


In [21]:
print(gdf.columns.tolist())  

['id', 'objectid', 'uuid', 'fitjuflokkur', 'stjornsyslusvaedi', 'nrsveitarfelags', 'sveitarfelag', 'upphafnotkunar', 'ath', 'vinnsluferlifitju', 'dagsheimildar', 'dagsinnsetningar', 'dagsleidrettingar', 'dagsuppfaerslu', 'nakvaemnixy', 'heimild', 'gagnaeigandi', 'gagnasafn', 'geometry']


In [22]:
# some muncipalities have more than one polygon/row, so we need to dissolve them

In [23]:
# using the make valid function from shapely for this
gdf['geometry'] = gdf['geometry'].apply(make_valid)
gdf_dissolved = gdf.dissolve(by='sveitarfelag').reset_index()
print(gdf_dissolved.shape)

(62, 19)


In [24]:
# since now we have 62 muncipalities, but the data from statistics Iceland has 63, I will compare the difference

In [25]:
# get all the municipalities from the file with the geoinformation
muncips_geo_file = sorted(gdf_dissolved['sveitarfelag'].tolist())

In [26]:
# then get all the municipalities from the data from Statistics Iceland
muncips_iceland_statistics = sorted(overnight_df_2024_long['region'].tolist())

In [27]:
list(set(muncips_geo_file) - set(muncips_iceland_statistics))

['Kópavogsbær',
 'Hveragerðisbær',
 'Seltjarnarnesbær',
 'Reykjavíkurborg',
 'Hafnarfjarðarkaupstaður',
 'Akureyrarbær',
 'Grindavíkurbær',
 'Vestmannaeyjabær',
 'Bolungarvíkurkaupstaður']

In [28]:
list(set(muncips_iceland_statistics) - set(muncips_geo_file))

['Vestmannaeyjar',
 'Bolungarvík',
 'Akureyri',
 'Seltjarnarnes',
 'Hveragerði',
 'Kópavogur',
 'Allt landið',
 'Grindavík',
 'Reykjavík',
 'Hafnarfjörður']

In [29]:
# looks like there are some differences in how things are spelled, we need to fix that to join them
# additional the statistics file contains one summary row (Allt landið) which I will remove

In [30]:
# removing the summary row from the Iceland Statistics list first
muncips_iceland_statistics = [m for m in muncips_iceland_statistics if m != 'Allt landið']

# Mapping geo file names to statistics names manually
name_mapping = {
    'Reykjavíkurborg': 'Reykjavík',
    'Kópavogsbær': 'Kópavogur',
    'Hafnarfjarðarkaupstaður': 'Hafnarfjörður',
    'Akureyrarbær': 'Akureyri',
    'Bolungarvíkurkaupstaður': 'Bolungarvík',
    'Grindavíkurbær': 'Grindavík',
    'Hveragerðisbær': 'Hveragerði',
    'Vestmannaeyjabær': 'Vestmannaeyjar',
    'Seltjarnarnesbær': 'Seltjarnarnes',
}

gdf_dissolved['sveitarfelag'] = gdf_dissolved['sveitarfelag'].replace(name_mapping)

# Verify no mismatches remain
print(set(gdf_dissolved['sveitarfelag']) - set(muncips_iceland_statistics)) 

set()


In [31]:
# since the set is empty, there are no more mismatches

In [32]:
print(len(gdf_dissolved))
print(len(overnight_df_2024_long))

62
63


In [33]:
# still need to remove the extra row from the dataframe
overnight_df_2024_long = overnight_df_2024_long.drop([0, 0])

In [34]:
len(overnight_df_2024_long)

62

In [35]:
overnight_df_2024_long.head()

,region,nights
1,Reykjavík,3218190.0
2,Kópavogur,85385.0
3,Hafnarfjörður,94045.0
4,Mosfellsbær,34910.0
5,Kjósarhreppur,NaN


### Merge the Dataframes

In [36]:
gdf_joined = gdf_dissolved.merge(overnight_df_2024_long, left_on="sveitarfelag", right_on="region", how="left")

In [37]:
gdf_joined.head(2)

,sveitarfelag,geometry,id,objectid,uuid,fitjuflokkur,stjornsyslusvaedi,nrsveitarfelags,upphafnotkunar,ath,...,dagsheimildar,dagsinnsetningar,dagsleidrettingar,dagsuppfaerslu,nakvaemnixy,heimild,gagnaeigandi,gagnasafn,region,nights
0,Akraneskaupstaður,GEOMETRYCOLLECTION (POLYGON ((-22.10030 64.305...,mork_sveitarf_flakar.42,42,7ccd7161-27ec-4c4b-9db0-08cc66aea592,101,4,3000,1944-01-01,,...,None,2003-01-01,2022-06-07,2024-06-26,50,LMÍ,LMÍ,"IS 50V, útgáfa 26.06.2024",Akraneskaupstaður,34131.0
1,Akureyri,GEOMETRYCOLLECTION (POLYGON ((-18.05270 65.650...,mork_sveitarf_flakar.29,29,39f3d7f1-5870-4632-eb95-87aba12425fd,101,4,6000,2020-01-24,,...,None,2003-01-01,2020-01-24,2024-06-26,50,AMS/DMA,LMÍ,"IS 50V, útgáfa 26.06.2024",Akureyri,335053.0


In [38]:
gdf_joined.columns

Index(['sveitarfelag', 'geometry', 'id', 'objectid', 'uuid', 'fitjuflokkur',
       'stjornsyslusvaedi', 'nrsveitarfelags', 'upphafnotkunar', 'ath',
       'vinnsluferlifitju', 'dagsheimildar', 'dagsinnsetningar',
       'dagsleidrettingar', 'dagsuppfaerslu', 'nakvaemnixy', 'heimild',
       'gagnaeigandi', 'gagnasafn', 'region', 'nights'],
      dtype='object')

In [39]:
# keep only the columns relevant to me
gdf_clean = gdf_joined[['sveitarfelag', 'nrsveitarfelags', 'stjornsyslusvaedi', 'nights', 'geometry']]

In [40]:
# check and adjust CRS
gdf_clean.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [41]:
# change to Icelandic crs
gdf_clean = gdf_clean.to_crs(3057)

In [42]:
gdf_clean.crs

<Projected CRS: EPSG:3057>
Name: ISN93 / Lambert 1993
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: Iceland - onshore and offshore.
- bounds: (-30.87, 59.96, -5.55, 69.59)
Coordinate Operation:
- name: Iceland Lambert 1993
- method: Lambert Conic Conformal (2SP)
Datum: Islands Net 1993
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

### Prepare population dataframe for matching

In [43]:
# check if naming is different
print(set(gdf_clean['sveitarfelag']) - set(population_df_2024['municipality']))  # in geo but not in stats
print(set(population_df_2024['municipality']) - set(gdf_clean['sveitarfelag']))  # in stats but not in geo

{'Hörgársveit', 'Sveitarfélagið Vogar', 'Vestmannaeyjar', 'Borgarbyggð', 'Dalabyggð', 'Grýtubakkahreppur', 'Akureyri', 'Langanesbyggð', 'Kjósarhreppur', 'Fjarðabyggð', 'Mosfellsbær', 'Reykjanesbær', 'Snæfellsbær', 'Sveitarfélagið Hornafjörður', 'Kópavogur', 'Skagafjörður', 'Vopnafjarðarhreppur', 'Ásahreppur', 'Reykjavík', 'Svalbarðsstrandarhreppur', 'Hvalfjarðarsveit', 'Fljótsdalshreppur', 'Sveitarfélagið Stykkishólmur', 'Múlaþing', 'Seltjarnarnes', 'Grímsnes- og Grafningshreppur', 'Dalvíkurbyggð', 'Rangárþing eystra', 'Skaftárhreppur', 'Rangárþing ytra', 'Fjallabyggð', 'Reykhólahreppur', 'Árneshreppur', 'Bolungarvík', 'Sveitarfélagið Skagaströnd', 'Suðurnesjabær', 'Hveragerði', 'Sveitarfélagið Árborg', 'Súðavíkurhreppur', 'Tjörneshreppur', 'Grindavík', 'Flóahreppur', 'Vesturbyggð', 'Ísafjarðarbær', 'Grundarfjarðarbær', 'Sveitarfélagið Ölfus', 'Hafnarfjörður', 'Húnabyggð', 'Bláskógabyggð', 'Strandabyggð', 'Þingeyjarsveit', 'Norðurþing', 'Akraneskaupstaður', 'Eyjafjarðarsveit', 'Skeiða-

In [44]:
# the difference is the use of Iceland letters
# to avoid issues with processing, I will normalize the letters used to be only standard letters and not Icelandic ones

In [45]:
# define a function to do this
def normalize_icelandic(name):
    replacements = {
        'ð': 'd', 'Ð': 'D',
        'þ': 'th', 'Þ': 'Th',
        'æ': 'ae', 'Æ': 'Ae',
        'ö': 'o', 'Ö': 'O',
        'á': 'a', 'Á': 'A',
        'é': 'e', 'É': 'E',
        'í': 'i', 'Í': 'I',
        'ó': 'o', 'Ó': 'O',
        'ú': 'u', 'Ú': 'U',
        'ý': 'y', 'Ý': 'Y',
        'ó': 'o', 'ú': 'u',
    }
    for char, replacement in replacements.items():
        name = name.replace(char, replacement)
    return name

In [46]:
# Normalize both for matching
gdf_clean['sveitarfelag_normalized'] = gdf_clean['sveitarfelag'].apply(normalize_icelandic)
population_df_2024['municipality_normalized'] = population_df_2024['municipality'].apply(normalize_icelandic)

In [47]:
# check that there are no more differences
print(set(gdf_clean['sveitarfelag_normalized']) - set(population_df_2024['municipality_normalized']))
print(set(population_df_2024['municipality_normalized']) - set(gdf_clean['sveitarfelag_normalized']))

{'Bolungarvik', 'Vestmannaeyjar', 'Kopavogur', 'Grindavik', 'Akureyri', 'Seltjarnarnes', 'Hafnarfjordur', 'Hveragerdi', 'Reykjavik'}
{'Kopavogsbaer', 'Bolungarvikurkaupstadur', 'Hveragerdisbaer', 'Seltjarnarnesbaer', 'Grindavikurbaer', 'Hafnarfjardarkaupstadur', 'Akureyrarbaer', 'Vestmannaeyjabaer', 'Reykjavikurborg'}


In [48]:
# still a few mismatches, as previously with different naming

# map according 
suffix_mapping = {
    'Reykjavik': 'Reykjavikurborg',
    'Kopavogur': 'Kopavogsbaer',
    'Hafnarfjordur': 'Hafnarfjardarkaupstadur',
    'Akureyri': 'Akureyrarbaer',
    'Bolungarvik': 'Bolungarvikurkaupstadur',
    'Grindavik': 'Grindavikurbaer',
    'Hveragerdi': 'Hveragerdisbaer',
    'Vestmannaeyjar': 'Vestmannaeyjabaer',
    'Seltjarnarnes': 'Seltjarnarnesbaer',
}

gdf_clean['sveitarfelag_normalized'] = gdf_clean['sveitarfelag_normalized'].replace(suffix_mapping)

In [49]:
# check again
print(set(gdf_clean['sveitarfelag_normalized']) - set(population_df_2024['municipality_normalized']))
print(set(population_df_2024['municipality_normalized']) - set(gdf_clean['sveitarfelag_normalized']))

set()
set()


In [50]:
# join the two dataframes

iceland_overnight_df_geo = gdf_clean.merge(population_df_2024, left_on="sveitarfelag_normalized", right_on="municipality_normalized", how="left")

In [51]:
iceland_overnight_df_geo.head(2)

,sveitarfelag,nrsveitarfelags,stjornsyslusvaedi,nights,geometry,sveitarfelag_normalized,municipality,population,municipality_normalized
0,Akraneskaupstaður,3000,4,34131.0,GEOMETRYCOLLECTION (POLYGON ((350015.715 42622...,Akraneskaupstadur,Akraneskaupstadur,8071,Akraneskaupstadur
1,Akureyri,6000,4,335053.0,GEOMETRYCOLLECTION (POLYGON ((543597.990 57279...,Akureyrarbaer,Akureyrarbaer,19812,Akureyrarbaer


In [53]:
iceland_overnight_df_geo = iceland_overnight_df_geo[['sveitarfelag_normalized', 'nrsveitarfelags', 'stjornsyslusvaedi', 
                                                     'nights', 'population', 'geometry']]

In [54]:
iceland_overnight_df_geo = iceland_overnight_df_geo.rename(columns = {'sveitarfelag_normalized': 'municipality'})

### Add population and area data

In [55]:
# first add area, based on geometry
iceland_overnight_df_geo["area_km2"] = iceland_overnight_df_geo.geometry.area / 1_000_000

In [56]:
# then calculate stays per population
iceland_overnight_df_geo['stays_per_pop'] = iceland_overnight_df_geo['nights']/iceland_overnight_df_geo['population']

In [57]:
# next calculate stays per area
iceland_overnight_df_geo['stays_per_area'] = iceland_overnight_df_geo['nights']/iceland_overnight_df_geo['area_km2']

In [58]:
iceland_overnight_df_geo.head()

,municipality,nrsveitarfelags,stjornsyslusvaedi,nights,population,geometry,area_km2,stays_per_pop,stays_per_area
0,Akraneskaupstadur,3000,4,34131.0,8071,GEOMETRYCOLLECTION (POLYGON ((350015.715 42622...,10.229906,4.228844,3336.394327
1,Akureyrarbaer,6000,4,335053.0,19812,GEOMETRYCOLLECTION (POLYGON ((543597.990 57279...,135.659164,16.911619,2469.814723
2,Blaskogabyggd,8721,4,207200.0,1322,"POLYGON ((406946.688 407295.729, 406552.260 40...",3300.001558,156.732224,62.787849
3,Bolungarvikurkaupstadur,4100,4,84853.0,989,"POLYGON ((312926.657 632864.473, 312889.139 63...",108.097114,85.796764,784.970081
4,Borgarbyggd,3609,4,254837.0,4100,GEOMETRYCOLLECTION (POLYGON ((337604.515 44913...,4926.804871,62.155366,51.724598


In [59]:
iceland_overnight_df_geo.crs

<Projected CRS: EPSG:3057>
Name: ISN93 / Lambert 1993
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: Iceland - onshore and offshore.
- bounds: (-30.87, 59.96, -5.55, 69.59)
Coordinate Operation:
- name: Iceland Lambert 1993
- method: Lambert Conic Conformal (2SP)
Datum: Islands Net 1993
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [60]:
# export to be used for choropleth maps and in routing
iceland_overnight_df_geo.to_file('iceland_overnight_geo_df.geojson', driver='GeoJSON')
print("Saved!")

Saved!
